# 04 — MEMO: evaluación con ejecutivo vía Ollama

Evalúa el MEMORY entrenado en el notebook 03 con el protocolo MEMO (descomposición → sub-respuestas de MEMORY → síntesis) sobre las mismas 12 preguntas y con el mismo scorer (`memo/evaluate.py`).

**Diseño Ollama:** el **EXECUTIVE** corre vía **ChatOllama** contra el servidor local (parámetro `EXEC_OLLAMA_MODEL`: `qwen2.5:1.5b` para paridad con el run 2 en CPU, `qwen2.5:7b` para el run con modelo más grande). El **MEMORY** es el modelo entrenado (1.5B + adaptador LoRA del notebook 03) y se carga con Hugging Face en GPU — entrenar y servir un LoRA no se puede hacer vía Ollama, así que MEMORY sigue siendo HF. El EXECUTIVE nunca ve el corpus: solo las sub-respuestas de MEMORY.

**Prerrequisito:** notebook 03 (adaptador en `memo/memory_model_colab`).

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

# --- Parametros ---
MEMORY_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # base de MEMORY (misma que el entreno)
ADAPTER = REPO + "/memo/memory_model_colab"    # adaptador del notebook 03
EXEC_OLLAMA_MODEL = "qwen2.5:1.5b"   # ejecutivo: 1.5b (paridad run 2) o qwen2.5:7b (mas grande)
RUN_CONTROL = True                   # tambien evalua la base sin entrenar (control)
GENERATE_EXPANDED = False            # True = genera set de evaluacion ampliado (solo genera; NO evalua sin revision)
EXPANDED_EVAL = REPO + "/memo/eval_questions_expanded.json"
TRACE_OUT = REPO + "/memo/run_trace_colab.jsonl"
TRACE_CONTROL_OUT = REPO + "/memo/run_trace_colab_control.jsonl"

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

## Datos procesados: primero en Drive, si no, súbelos

Este notebook necesita `pipeline/data/chunks.jsonl` (las 151 páginas del informe extraídas). El flujo es:

1. **Buscarlo en Drive** (ruta estándar, bajo el repo clonado).
2. **Si no está, subirlo** con el widget de Colab de la celda siguiente: el archivo es `chunks.jsonl`. Lo consigues de una corrida anterior en Drive o generándolo del repo con `make extract` (`python pipeline/extract.py`).
3. **Solo si no lo subes**, generarlo aquí corriendo `extract.py` (descarga el PDF ~19 MB y extrae las 151 páginas, ~1–2 min). Es el último recurso, no el camino principal.

`triples_seed.json` y `eval_questions.json` vienen con el clon del repo: no hace falta subirlos.

In [ ]:
import json, subprocess, sys
from pathlib import Path

CHUNKS = REPO + "/pipeline/data/chunks.jsonl"
DATA_DIR = REPO + "/pipeline/data"


def _chunks_ok(path):
    try:
        with open(path) as f:
            first = f.readline()
            n = 1 + sum(1 for _ in f)
        row = json.loads(first)
        return n >= 100 and isinstance(row.get("text"), str) and len(row["text"]) > 0
    except Exception:
        return False


if _chunks_ok(CHUNKS):
    print(f"datos procesados OK en Drive: {CHUNKS}")
else:
    print("chunks.jsonl NO está en Drive.")
    print("Súbelo con el widget de abajo (botón 'Elegir archivos').")
    print("Qué subir: chunks.jsonl — las 151 páginas del informe FEMSA 2024.")
    print("Dónde conseguirlo: de una corrida anterior en Drive, o del repo con `make extract`.")
    print("Si no lo subes, el notebook lo genera con extract.py (último recurso).")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = Path(DATA_DIR) / Path(name).name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"subido: {dest} ({len(data)} bytes)")
    if not _chunks_ok(CHUNKS) and len(uploaded) == 1:
        # El archivo subido puede venir con otro nombre: probar como chunks.jsonl.
        only = next(iter(uploaded.values()))
        Path(CHUNKS).write_bytes(only)
        print("probando el archivo subido como chunks.jsonl...")
    if _chunks_ok(CHUNKS):
        print("datos listos por subida.")
    else:
        print("Sin subida válida: último recurso, generar con extract.py...")
        print("(descarga el PDF ~19 MB y extrae las 151 páginas, ~1-2 min)")
        try:
            import pypdf
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "pypdf>=7.1.1"], check=True)
        r = subprocess.run([sys.executable, "extract.py"], cwd=REPO + "/pipeline")
        print("extract.py exit:", r.returncode)
        assert _chunks_ok(CHUNKS), "No se pudo obtener chunks.jsonl."
    print(f"chunks.jsonl listo: {CHUNKS}")

In [ ]:
%pip install -q "torch>=2.15.0" transformers==5.24.1 accelerate>=1.16.0 peft>=0.22.0 numpy>=2.5.5 langchain-ollama

## Ollama (setup compacto e idempotente)

Este notebook usa modelos intermedios vía **Ollama** corriendo en la propia instancia de Colab (un solo servidor, cambio de modelo con un parámetro, sin cargar pesos en cada notebook). Las celdas son idempotentes: verifican si el binario existe, si el servidor ya responde y si el modelo ya está bajado antes de hacer nada. La versión completa, con la explicación y las fuentes, está en `00_ollama_setup.ipynb`.

In [ ]:
%%bash
if command -v ollama >/dev/null 2>&1; then
  echo "ollama ya instalado: $(ollama --version)"
else
  echo "instalando Ollama (primero zstd, que el instalador exige y Colab no trae)..."
  apt-get -qq update && apt-get -qq install -y zstd pciutils
  curl -fsSL https://ollama.com/install.sh | sh
  ollama --version
fi

In [ ]:
import os, subprocess, time, urllib.request

def ollama_up():
    try:
        with urllib.request.urlopen("http://localhost:11434", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

if ollama_up():
    print("servidor Ollama ya responde en http://localhost:11434")
else:
    print("arrancando `ollama serve` en segundo plano...")
    out = open("ollama.log", "w")
    err = open("ollama_error.log", "w")
    subprocess.Popen(["nohup", "ollama", "serve"], stdout=out, stderr=err,
                     preexec_fn=os.setsid)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(2)
    print("Ollama responde en localhost:11434:", ollama_up())
    if not ollama_up():
        print("Revisa ollama_error.log; el servidor no levanto.")

In [ ]:
%%bash
if ollama show "qwen2.5:1.5b" >/dev/null 2>&1; then
  echo "qwen2.5:1.5b ya descargado"
else
  echo "bajando qwen2.5:1.5b..."
  ollama pull "qwen2.5:1.5b"
fi
if ollama show "qwen2.5:7b" >/dev/null 2>&1; then
  echo "qwen2.5:7b ya descargado"
else
  echo "bajando qwen2.5:7b..."
  ollama pull "qwen2.5:7b"
fi
ollama list

In [ ]:
from langchain_ollama import ChatOllama

_llm = ChatOllama(model=EXEC_OLLAMA_MODEL, base_url="http://localhost:11434", temperature=0)
print(_llm.invoke("Responde con una sola palabra: ¿cuánto es 2+2?").content)
print("ChatOllama OK contra", EXEC_OLLAMA_MODEL)

## Prerrequisito: el adaptador MEMORY del notebook 03

Sin el adaptador entrenado no hay nada que evaluar. Esta celda falla con un mensaje claro si no existe: corre el notebook 03 primero.

In [ ]:
import os
assert os.path.isdir(ADAPTER), (
    "FALTA el adaptador MEMORY: " + ADAPTER + "\n"
    "Lo produce el notebook 03 (MEMO entrenamiento). Corre 03 primero; "
    "sin adaptador entrenado, la evaluación no tiene sentido."
)
print("adaptador MEMORY OK:", ADAPTER)

## 1. Cargar MEMORY (HF, GPU) y EXECUTIVE (Ollama)

MEMORY = base 1.5B + adaptador LoRA del notebook 03 (o la base sola si `RUN_CONTROL`). EXECUTIVE = ChatOllama contra el servidor local. Se reutilizan los prompts y la descomposición por reglas de `memo/executive_v2.py`.

In [ ]:
import json, sys, torch
sys.path.insert(0, REPO + "/memo")
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from executive_v2 import DECOMP_SYSTEM, DECOMP_EXAMPLES, rule_decompose

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

exec_llm = ChatOllama(model=EXEC_OLLAMA_MODEL, base_url="http://localhost:11434",
                      temperature=0, num_predict=150)
print("EXECUTIVE (Ollama):", EXEC_OLLAMA_MODEL)


def chat_ollama(system, user):
    msgs = [SystemMessage(content=system), HumanMessage(content=user)]
    return exec_llm.invoke(msgs).content.strip()


mem_tok = AutoTokenizer.from_pretrained(MEMORY_MODEL)
print(f"cargando MEMORY {MEMORY_MODEL} (bf16, device_map=auto)...", flush=True)
_mem_base = AutoModelForCausalLM.from_pretrained(MEMORY_MODEL,
                                                 torch_dtype=torch.bfloat16,
                                                 device_map="auto")
print(f"cargando adaptador {ADAPTER} ...", flush=True)
mem_model = PeftModel.from_pretrained(_mem_base, ADAPTER)
mem_model.eval()
print("MEMORY lista: base + adaptador")

## 2. Protocolo MEMO (adaptado de `executive_v2.py`)

Mismo protocolo que el run 2: descomposición por LLM (ahora vía Ollama; si falla el parseo, cae a la regla documentada), sub-respuestas de MEMORY (greedy, cortas) y síntesis por el EXECUTIVE. La traza guarda el método de descomposición (`llm` o `rule`) por pregunta, como en el original.

In [ ]:
def llm_decompose(q):
    out = chat_ollama(DECOMP_SYSTEM, DECOMP_EXAMPLES + f"Pregunta: {q}\n")
    lines = [l.strip().lstrip("0123456789.-) ").strip()
             for l in out.splitlines() if l.strip().endswith("?")]
    lines = [l for l in lines if len(l) > 10][:3]
    return lines or None


def memory_answer(subq):
    system = ("Responde con el dato exacto del informe anual de FEMSA 2024, "
              "en una frase corta. Si no sabes el dato, di 'No lo sé'.")
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": subq}]
    text = mem_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = mem_tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = mem_model.generate(**ids, max_new_tokens=60, do_sample=False,
                                 pad_token_id=mem_tok.eos_token_id)
    return mem_tok.decode(out[0][ids["input_ids"].shape[1]:],
                          skip_special_tokens=True).strip()


def synthesize(q, sub_answers):
    context = "\n".join(f"- {s}: {a}" for s, a in sub_answers)
    system = ("Responde la pregunta usando SOLO la información siguiente, "
              "que viene del modelo de memoria. No inventes cifras. "
              "Si la información no alcanza, dilo.")
    user = f"Información del modelo de memoria:\n{context}\n\nPregunta: {q}"
    return chat_ollama(system, user)


def run_memo(q):
    subs = llm_decompose(q)
    method = "llm"
    if not subs:
        subs = rule_decompose(q)
        method = "rule"
    sub_answers = [(s, memory_answer(s)) for s in subs]
    final = synthesize(q, sub_answers)
    return {"question": q, "decomposition": method,
            "exec_model": f"ollama {EXEC_OLLAMA_MODEL}",
            "memory_model": MEMORY_MODEL,
            "sub_questions": subs,
            "sub_answers": [{"sub_question": s, "memory_answer": a}
                            for s, a in sub_answers],
            "final_answer": final}

## 3. Correr las 12 preguntas (MEMORY entrenado)

In [ ]:
import json

eval_qs = json.load(open(REPO + "/memo/eval_questions.json"))["questions"]
with open(TRACE_OUT, "w") as f:
    for item in eval_qs:
        r = run_memo(item["question"])
        r["id"] = item["id"]
        r["level"] = item["level"]
        r["control"] = False
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"{item['id']} [{r['decomposition']}] -> {r['final_answer'][:120]}", flush=True)
print(f"traza -> {TRACE_OUT}")

## 4. Control (base sin entrenar, mismo ejecutivo)

Si `RUN_CONTROL` es True, repite las 12 con MEMORY = base sin adaptador. El control mide cuánto viene del entrenamiento y cuánto del ejecutivo: en el run 2 el control sacó 0/12 con el mismo ejecutivo 1.5B.

In [ ]:
if RUN_CONTROL:
    print("cargando MEMORY control (base sin adaptador)...", flush=True)
    _base = AutoModelForCausalLM.from_pretrained(MEMORY_MODEL,
                                                 torch_dtype=torch.bfloat16,
                                                 device_map="auto")
    _base.eval()
    mem_model = _base
    eval_qs = json.load(open(REPO + "/memo/eval_questions.json"))["questions"]
    with open(TRACE_CONTROL_OUT, "w") as f:
        for item in eval_qs:
            r = run_memo(item["question"])
            r["id"] = item["id"]
            r["level"] = item["level"]
            r["control"] = True
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
            print(f"{item['id']} [{r['decomposition']}] -> {r['final_answer'][:120]}", flush=True)
    print(f"traza control -> {TRACE_CONTROL_OUT}")
else:
    print("RUN_CONTROL=False: control omitido.")

## 5. Puntuar con el mismo scorer

Corre `memo/evaluate.py` tal cual (mismo scorer, misma regla Q3 con 1,596, mismas keywords). Compara con el run 2 en CPU (2/12) y el run 1 (0/12).

In [ ]:
import subprocess, sys
for trace, out in [(TRACE_OUT, REPO + "/memo/results_colab.md"),
                   (TRACE_CONTROL_OUT, REPO + "/memo/results_colab_control.md")]:
    if RUN_CONTROL or "control" not in trace:
        r = subprocess.run([sys.executable, "evaluate.py", "--trace", trace,
                            "--eval", REPO + "/memo/eval_questions.json",
                            "--qa", REPO + "/memo/reflection_qa_v3.jsonl",
                            "--out", out], cwd=REPO + "/memo")
        print("exit:", r.returncode, "->", out)

## 6. (Opcional) Generar un set de evaluación ampliado

Con `GENERATE_EXPANDED = True` esta celda genera preguntas nuevas a partir de las tripletas semilla (hechos ya verificados) con plantillas por tipo de tripleta. **Cada cifra esperada se verifica por grep literal en el corpus (`chunks.jsonl`): lo que no aparece literal, se descarta — nunca se adivina.** También se descartan las que se parecen demasiado (Jaccard ≥ 0.80) a las 12 originales o entre sí. La salida lleva `"needs_review": true`: hay que revisar las preguntas a mano antes de evaluar con ellas (cambia `EVAL_FILE` en la celda 3 y vuelve a correr). Por defecto está apagado.

In [ ]:
import json, re, sys
sys.path.insert(0, REPO + "/memo")
from evaluate import digits

if not GENERATE_EXPANDED:
    print("GENERATE_EXPANDED=False: no se genera nada. Ponlo en True para generar el set.")
else:
    triples = json.load(open(REPO + "/pipeline/triples_seed.json"))["triples"]
    chunks = [json.loads(l) for l in open(REPO + "/pipeline/data/chunks.jsonl")]
    corpus = "\n".join(c["text"] for c in chunks)
    orig = json.load(open(REPO + "/memo/eval_questions.json"))["questions"]

    def tokens(q):
        import unicodedata
        q = unicodedata.normalize("NFKD", q.lower())
        q = "".join(c for c in q if not unicodedata.combining(c))
        return set(re.findall(r"[a-z0-9]+", q))

    def jaccard(a, b):
        return len(a & b) / len(a | b) if a | b else 0.0

    def literal_present(text, expected):
        e = str(expected).replace(",", "")
        if "." in e:
            return e in text or e.replace(".", ",") in text
        try:
            grouped = f"{int(e):,}"
        except ValueError:
            return e in text
        return grouped in text or e in text

    orig_toks = [tokens(q["question"]) for q in orig]
    cands = []
    for t in triples:
        subj, rel, obj = t.get("subject"), t.get("relation"), t.get("object")
        valor, var = t.get("valor"), t.get("variacion_pct")
        if valor is None:
            continue
        if rel == "reporta" and obj == "Ingresos Totales":
            q = f"¿Cuáles fueron los ingresos totales de {subj} en 2024?"
            exp_nums = [str(valor)] + ([str(var)] if var is not None else [])
            exp = f"{valor} (Ps. millones)" + (f", +{var}%" if var is not None else "")
        elif rel == "mide" and obj == "Usuarios Activos":
            q = f"¿Cuántos usuarios activos tuvo {subj} en 2024?"
            exp_nums = [str(valor)] + ([str(var)] if var is not None else [])
            exp = f"{valor} millones" + (f" (+{var}%)" if var is not None else "")
        elif rel == "mide" and obj == "Tiendas":
            q = f"¿Cuántas tiendas tenía {subj} al cierre de 2024?"
            exp_nums = [str(valor)]
            exp = f"{valor} tiendas"
        elif rel == "mide" and obj == "Estaciones":
            q = f"¿Cuántas estaciones tenía OXXO GAS al cierre de 2024?"
            exp_nums = [str(valor)]
            exp = f"{valor} estaciones"
        else:
            continue
        cands.append({"question": q, "expected": exp, "expected_numbers": exp_nums,
                      "source_triple": {k: t.get(k) for k in ("subject", "relation", "object", "valor", "variacion_pct")}})

    kept, dropped = [], []
    seen_q = set()
    for c in cands:
        key = tokens(c["question"])
        if c["question"] in seen_q:
            dropped.append((c["question"], "duplicada"))
            continue
        if max((jaccard(key, et) for et in orig_toks), default=0) >= 0.80:
            dropped.append((c["question"], "demasiado parecida a una de las 12"))
            continue
        missing = [e for e in c["expected_numbers"] if not literal_present(corpus, e)]
        if missing:
            dropped.append((c["question"], f"cifra no verificada por grep literal: {missing}"))
            continue
        seen_q.add(c["question"])
        kept.append(c)

    out = {"needs_review": True,
           "note": "Generado por plantillas desde triples_seed.json; cada cifra verificada por grep literal en chunks.jsonl. REVISAR A MANO antes de evaluar.",
           "questions": [{"id": f"X{i+1}", "level": 1, **c} for i, c in enumerate(kept)]}
    json.dump(out, open(EXPANDED_EVAL, "w"), ensure_ascii=False, indent=2)
    print(f"candidatas: {len(cands)} | aceptadas: {len(kept)} | descartadas: {len(dropped)}")
    for q, why in dropped:
        print(f"  DESCARTADA: {q} ({why})")
    for c in kept:
        print(f"  X: {c['question']} -> {c['expected']}")
    print(f"-> {EXPANDED_EVAL} (needs_review=true: revisa antes de usar)")

## Guardar en Drive (final)

Todo lo que produjo este notebook vive bajo el repo clonado en Drive (`REPO`), así que sobrevive aunque la sesión de Colab se corte. Esta celda lo verifica y lo lista: si algo falta, lo dice aquí, no después.

In [ ]:
import os, shutil, time


def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.0f} {u}"
        n /= 1024
    return f"{n:.1f} GB"


def _list_dir(path):
    total, nfiles = 0, 0
    for root, _dirs, files in os.walk(path):
        for fn in sorted(files):
            p = os.path.join(root, fn)
            sz = os.path.getsize(p)
            total += sz
            nfiles += 1
            print(f"    {os.path.relpath(p, REPO)} ({_human(sz)})")
    return nfiles, total


# Logs de la sesion (viven en /content, fuera de Drive): copiarlos al repo.
for _log in ["ollama.log", "ollama_error.log", "relate.log"]:
    if os.path.isfile(_log):
        shutil.copy(_log, os.path.join(REPO, _log))
        print(f"log copiado a Drive: {_log}")

ENTRIES = [('memo/run_trace_colab.jsonl', 'Traza MEMO Colab (12 preguntas)', True), ('memo/run_trace_colab_control.jsonl', 'Traza control (base sin entrenar)', False), ('memo/results_colab.md', 'Resultados MEMO Colab', True), ('memo/results_colab_control.md', 'Resultados control', False), ('memo/eval_questions_expanded.json', 'Set de evaluación ampliado (si se generó; needs_review=true)', False)]
total_bytes, missing = 0, []
for rel, label, required in ENTRIES:
    p = os.path.join(REPO, rel)
    if os.path.isdir(p):
        nf, tb = _list_dir(p)
        total_bytes += tb
        print(f"{label}: directorio, {nf} archivos, {_human(tb)}")
    elif os.path.isfile(p):
        sz = os.path.getsize(p)
        total_bytes += sz
        mt = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
        print(f"{label}: {rel} ({_human(sz)}, {mt})")
    else:
        status = "FALTA (obligatorio)" if required else "pendiente (opcional)"
        print(f"{label}: {status}: {rel}")
        if required:
            missing.append(rel)
print(f"\nGUARDADO EN DRIVE: {_human(total_bytes)} verificados bajo {REPO}")
if missing:
    print("OJO, faltan obligatorios:", missing)